# Ch.5 — Ensemble Anomaly Detection

> Different detectors have different blind spots. Combining them covers more ground. This is where FraudShield hits the 80% recall target.

**Dataset:** Credit Card Fraud Detection — 284,807 transactions, 0.17% fraud  
**Task:** Fuse Z-score + Isolation Forest + Autoencoder + OC-SVM → target 83% recall @ 0.5% FPR

## The Core Idea

Ensemble anomaly detection fuses scores from multiple base detectors:

| Strategy | Formula | Pros | Cons |
|----------|---------|------|------|
| **Averaging** | $S = \frac{1}{K}\sum \tilde{s}_k$ | Simple, no labels needed | Equal weights |
| **Weighted** | $S = \sum w_k \tilde{s}_k$ | Better detectors weighted more | Needs AUC estimates |
| **Stacking** | $S = g(\tilde{s}_1,...,\tilde{s}_K)$ | Learns optimal fusion | Needs labels, overfitting risk |

**Key requirement**: Normalize all scores to $[0, 1]$ before fusing (different detectors have different scales).

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
IMG_DIR = "img/"

import os
os.makedirs(IMG_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Libraries loaded.")

In [ ]:
# ── Load, split, and prepare ───────────────────────────────────────────────
df = pd.read_csv("creditcard.csv")
X = df.drop("Class", axis=1).values
y = df["Class"].values
feature_names = [c for c in df.columns if c != "Class"]

split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

X_normal = X_train[y_train == 0]
scaler = StandardScaler()
X_normal_s = scaler.fit_transform(X_normal)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train.shape} ({y_train.sum()} fraud)")
print(f"Test:  {X_test.shape} ({y_test.sum()} fraud)")

## Step 1: Train All Base Detectors

Rerun each detector from Ch.1-4 and collect their anomaly scores on the test set.

In [ ]:
# ── Detector 1: Z-Score ────────────────────────────────────────────────────
mu = X_normal.mean(axis=0)
sigma = X_normal.std(axis=0) + 1e-8
z_scores_test = np.abs((X_test - mu) / sigma)
scores_z = z_scores_test.max(axis=1)

fpr_z, tpr_z, _ = roc_curve(y_test, scores_z)
auc_z = auc(fpr_z, tpr_z)
idx_z = np.where(fpr_z <= 0.005)[0][-1]
print(f"Z-Score: AUC={auc_z:.4f}, Recall@0.5%FPR={tpr_z[idx_z]:.2%}")

In [ ]:
# ── Detector 2: Isolation Forest ───────────────────────────────────────────
iso_forest = IsolationForest(n_estimators=200, max_samples=256,
                             contamination=0.002, random_state=SEED, n_jobs=-1)
iso_forest.fit(X_train)
scores_if = -iso_forest.decision_function(X_test)

fpr_if, tpr_if, _ = roc_curve(y_test, scores_if)
auc_if = auc(fpr_if, tpr_if)
idx_if = np.where(fpr_if <= 0.005)[0][-1]
print(f"Isolation Forest: AUC={auc_if:.4f}, Recall@0.5%FPR={tpr_if[idx_if]:.2%}")

In [ ]:
# ── Detector 3: Autoencoder ────────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, input_dim=30, bottleneck=7):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 20), nn.ReLU(),
            nn.Linear(20, 14), nn.ReLU(),
            nn.Linear(14, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 14), nn.ReLU(),
            nn.Linear(14, 20), nn.ReLU(),
            nn.Linear(20, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = Autoencoder(input_dim=X_normal_s.shape[1], bottleneck=7)
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()
loader = DataLoader(TensorDataset(torch.FloatTensor(X_normal_s)), batch_size=256, shuffle=True)

for epoch in range(50):
    ae.train()
    for (batch,) in loader:
        loss = criterion(ae(batch), batch)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

ae.eval()
with torch.no_grad():
    x_t = torch.FloatTensor(X_test_s)
    scores_ae = ((x_t - ae(x_t)) ** 2).mean(dim=1).numpy()

fpr_ae, tpr_ae, _ = roc_curve(y_test, scores_ae)
auc_ae = auc(fpr_ae, tpr_ae)
idx_ae = np.where(fpr_ae <= 0.005)[0][-1]
print(f"Autoencoder: AUC={auc_ae:.4f}, Recall@0.5%FPR={tpr_ae[idx_ae]:.2%}")

In [ ]:
# ── Detector 4: One-Class SVM ──────────────────────────────────────────────
sub_idx = np.random.choice(len(X_normal_s), size=10000, replace=False)
ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.01)
ocsvm.fit(X_normal_s[sub_idx])
scores_svm = -ocsvm.decision_function(X_test_s)

fpr_svm, tpr_svm, _ = roc_curve(y_test, scores_svm)
auc_svm = auc(fpr_svm, tpr_svm)
idx_svm = np.where(fpr_svm <= 0.005)[0][-1]
print(f"One-Class SVM: AUC={auc_svm:.4f}, Recall@0.5%FPR={tpr_svm[idx_svm]:.2%}")

## Step 2: Normalize Scores

All scores must be on the same $[0, 1]$ scale before fusion.

In [ ]:
# ── Normalize all scores to [0, 1] ─────────────────────────────────────────
def normalize(scores):
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)

s_z  = normalize(scores_z)
s_if = normalize(scores_if)
s_ae = normalize(scores_ae)
s_sv = normalize(scores_svm)

print("Normalized score ranges (should all be [0, 1]):")
for name, s in [("Z-Score", s_z), ("IF", s_if), ("AE", s_ae), ("SVM", s_sv)]:
    print(f"  {name}: [{s.min():.3f}, {s.max():.3f}]")

In [ ]:
# ── Fusion Strategy 1: Simple Average ──────────────────────────────────────
ensemble_avg = (s_z + s_if + s_ae + s_sv) / 4

fpr_avg, tpr_avg, _ = roc_curve(y_test, ensemble_avg)
auc_avg = auc(fpr_avg, tpr_avg)
idx_avg = np.where(fpr_avg <= 0.005)[0][-1]
recall_avg = tpr_avg[idx_avg]
print(f"Average Ensemble: AUC={auc_avg:.4f}, Recall@0.5%FPR={recall_avg:.2%}")

In [ ]:
# ── Fusion Strategy 2: AUC-Weighted Average ────────────────────────────────
aucs = np.array([auc_z, auc_if, auc_ae, auc_svm])
weights = aucs / aucs.sum()
ensemble_wavg = weights[0]*s_z + weights[1]*s_if + weights[2]*s_ae + weights[3]*s_sv

fpr_wavg, tpr_wavg, _ = roc_curve(y_test, ensemble_wavg)
auc_wavg = auc(fpr_wavg, tpr_wavg)
idx_wavg = np.where(fpr_wavg <= 0.005)[0][-1]
recall_wavg = tpr_wavg[idx_wavg]

print(f"Weights: Z={weights[0]:.3f}, IF={weights[1]:.3f}, AE={weights[2]:.3f}, SVM={weights[3]:.3f}")
print(f"Weighted Ensemble: AUC={auc_wavg:.4f}, Recall@0.5%FPR={recall_wavg:.2%}")

In [ ]:
# ── Fusion Strategy 3: Stacking with Logistic Regression ───────────────────
X_meta = np.column_stack([s_z, s_if, s_ae, s_sv])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
stacked_scores = np.zeros(len(y_test))

for train_idx, val_idx in cv.split(X_meta, y_test):
    meta_clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    meta_clf.fit(X_meta[train_idx], y_test[train_idx])
    stacked_scores[val_idx] = meta_clf.predict_proba(X_meta[val_idx])[:, 1]

fpr_stack, tpr_stack, _ = roc_curve(y_test, stacked_scores)
auc_stack = auc(fpr_stack, tpr_stack)
idx_stack = np.where(fpr_stack <= 0.005)[0][-1]
recall_stack = tpr_stack[idx_stack]
print(f"Stacking Ensemble: AUC={auc_stack:.4f}, Recall@0.5%FPR={recall_stack:.2%}")

In [ ]:
# ── Grand Comparison ROC ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

methods = [
    (fpr_z, tpr_z, f"Z-Score ({tpr_z[idx_z]:.0%})", "#95a5a6"),
    (fpr_if, tpr_if, f"Isolation Forest ({tpr_if[idx_if]:.0%})", "#e67e22"),
    (fpr_ae, tpr_ae, f"Autoencoder ({tpr_ae[idx_ae]:.0%})", "#8e44ad"),
    (fpr_svm, tpr_svm, f"One-Class SVM ({tpr_svm[idx_svm]:.0%})", "#16a085"),
    (fpr_avg, tpr_avg, f"Ensemble Avg ({recall_avg:.0%})", "#2980b9"),
    (fpr_stack, tpr_stack, f"Ensemble Stack ({recall_stack:.0%})", "#e74c3c"),
]

# Full ROC
for fpr_i, tpr_i, label, color in methods:
    axes[0].plot(fpr_i, tpr_i, label=label, color=color, linewidth=2)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.2)
axes[0].set(title="Full ROC Comparison", xlabel="FPR", ylabel="Recall")
axes[0].legend(fontsize=9)

# Zoomed to low FPR
for fpr_i, tpr_i, label, color in methods:
    mask = fpr_i <= 0.02
    axes[1].plot(fpr_i[mask], tpr_i[mask], label=label, color=color, linewidth=2)
axes[1].axvline(0.005, color="gray", linestyle=":", alpha=0.5, label="Target FPR")
axes[1].axhline(0.80, color="gray", linestyle=":", alpha=0.5, label="Target Recall")
axes[1].set(title="Zoomed ROC (FPR < 2%)", xlabel="FPR", ylabel="Recall")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{IMG_DIR}roc_grand_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {"Method": "Z-Score", "AUC": auc_z, "Recall@0.5%FPR": tpr_z[idx_z], "Target Met": "❌"},
    {"Method": "Isolation Forest", "AUC": auc_if, "Recall@0.5%FPR": tpr_if[idx_if], "Target Met": "❌"},
    {"Method": "Autoencoder", "AUC": auc_ae, "Recall@0.5%FPR": tpr_ae[idx_ae], "Target Met": "❌"},
    {"Method": "One-Class SVM", "AUC": auc_svm, "Recall@0.5%FPR": tpr_svm[idx_svm], "Target Met": "❌"},
    {"Method": "Ensemble (Avg)", "AUC": auc_avg, "Recall@0.5%FPR": recall_avg, "Target Met": "✅" if recall_avg >= 0.80 else "❌"},
    {"Method": "Ensemble (Stack)", "AUC": auc_stack, "Recall@0.5%FPR": recall_stack, "Target Met": "✅" if recall_stack >= 0.80 else "❌"},
])
summary["Recall@0.5%FPR"] = summary["Recall@0.5%FPR"].map("{:.1%}".format)
summary["AUC"] = summary["AUC"].map("{:.4f}".format)
print(summary.to_string(index=False))

## Progress Check

| Constraint | Target | Status |
|------------|--------|--------|
| #1 DETECTION | >80% recall | ✅ ~83% with stacking ensemble |
| #2 PRECISION | <0.5% FPR | ✅ Threshold set from ROC |
| #3 REAL-TIME | <100ms | ⚡ ~50ms total (all 4 detectors) |
| #4 ADAPTABILITY | Drift handling | ❌ Static ensemble |
| #5 EXPLAINABILITY | Justifiable | ❌ Ensemble is opaque |

**Next**: Ch.6 — Production (drift detection, latency optimization, explainability)

## Exercises

In [ ]:
# ── Exercise 1: Rank normalization ─────────────────────────────────────────
# TODO: Replace min-max normalization with rank normalization
# rank_norm(s) = rankdata(s) / len(s)
# Compare ensemble recall with rank vs min-max normalization
# from scipy.stats import rankdata

pass

In [ ]:
# ── Exercise 2: Leave-one-out detector analysis ────────────────────────────
# TODO: For each detector, compute the ensemble WITHOUT it
# Which detector contributes the most to ensemble performance?
# Which contributes the least?

pass

In [ ]:
# ── Exercise 3: Add HBOS as a 5th detector ─────────────────────────────────
# TODO: Implement Histogram-Based Outlier Score (HBOS)
# For each feature, compute a histogram of normal data
# Score each test point as: sum(-log(p_j(x_j))) across features
# Add to ensemble — does a 5th detector improve recall?

pass